In [103]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import copy
import torch
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.decomposition import PCA
import joblib

In [104]:
train_df=pd.read_csv("/kaggle/input/datasets/karimmahmoud09/integrated-multi-turn-dataset/train.csv")
val_df=pd.read_csv("/kaggle/input/datasets/karimmahmoud09/integrated-multi-turn-dataset/validation.csv")
test_df=pd.read_csv("/kaggle/input/datasets/karimmahmoud09/integrated-multi-turn-dataset/test.csv")

In [105]:
vector_list= list(str(col) for col in range(768+768))

In [106]:
train_vectors_df = train_df[vector_list]
val_vectors_df = val_df[vector_list]
test_vectors_df = test_df[vector_list]

In [107]:
train_features_df = train_df.drop(columns=vector_list)
val_features_df = val_df.drop(columns=vector_list)
test_features_df = test_df.drop(columns=vector_list)

train_features_df.drop(columns=["conv_id","turn_id"], inplace=True)
val_features_df.drop(columns=["conv_id","turn_id"], inplace=True)
test_features_df.drop(columns=["conv_id","turn_id"], inplace=True)

pca = PCA(n_components=0.95)
train_vectors_df = pca.fit_transform(train_vectors_df)
val_vectors_df = pca.transform(val_vectors_df)
test_vectors_df = pca.transform(test_vectors_df)

In [108]:
joblib.dump(pca, "pca_model.pkl")

['pca_model.pkl']

# dataset

In [109]:
class FusionDataset(Dataset):

    def __init__(self, vectors_df, features_df, label_col="label"):
        self.vectors = torch.tensor(vectors_df, dtype=torch.float32)

        self.features = torch.tensor(
            features_df.drop(columns=[label_col]).values,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            features_df[label_col].values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.vectors[idx], self.features[idx], self.labels[idx]

In [110]:
train_dataset = FusionDataset(train_vectors_df, train_features_df)
val_dataset = FusionDataset(val_vectors_df, val_features_df)
test_dataset = FusionDataset(test_vectors_df, test_features_df)

In [111]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

# model

In [120]:
class FusionNet(nn.Module):
    def __init__(self, vector_dim=360, feature_dim=20, num_classes=2, dropout=0.3):
        super().__init__()

        self.vector_branch = nn.Sequential(
            nn.Linear(vector_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.feature_branch = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(160, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, vectors, features):
        v = self.vector_branch(vectors)
        f = self.feature_branch(features)
        x = torch.cat([v, f], dim=1)
        out=self.classifier(x)
        return out

# overfit one batch

In [121]:
def overfit_one_batch(model, train_loader, device, epochs=500):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    vectors, features, labels = next(iter(train_loader))

    vectors = vectors.to(device).float()
    features = features.to(device).float()
    labels = labels.to(device).long()

    for epoch in range(epochs):
        optimizer.zero_grad()

        outputs = model(vectors, features)
        loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        preds = outputs.argmax(dim=1)
        acc = (preds == labels).float().mean().item()

        if epoch % 20 == 0 or acc == 1.0:
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | Acc: {acc:.4f}")

        if acc == 1.0:
            print("Successfully overfit one batch!")
            break

In [122]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionNet(vector_dim=train_vectors_df.shape[1] ,feature_dim=train_features_df.shape[1]-1, num_classes=2, dropout=0.0).to(device)
overfit_one_batch(model, train_loader, device, epochs=500)

Epoch    0 | Loss: 0.680262 | Acc: 0.6094
Epoch    9 | Loss: 0.332980 | Acc: 1.0000
Successfully overfit one batch!


# train loop

In [123]:
def train_model(model, train_loader, val_loader, device, epochs=50, patience=7, model_path="best_model.pth"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    best_f1 = 0.0
    best_weights = None
    wait = 0

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for vectors, features, labels in train_loader:
            vectors = vectors.to(device).float()
            features = features.to(device).float()
            labels = labels.to(device).long()
            optimizer.zero_grad()

            outputs = model(vectors, features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        preds = []
        targets = []

        with torch.no_grad():
            for vectors, features, labels in val_loader:
                vectors = vectors.to(device).float()
                features = features.to(device).float()
                labels = labels.to(device).long()

                outputs = model(vectors, features)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                pred = outputs.argmax(dim=1)

                preds.extend(pred.cpu().numpy())
                targets.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(targets, preds, average="macro")

        scheduler.step(val_f1)

        current_lr = optimizer.param_groups[0]["lr"]

        print(f"Epoch {epoch + 1:3d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val F1 {val_f1:.4f} | LR {current_lr:.6f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            wait = 0
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, model_path)
            print("Validation F1 improved -> model saved.")
        else:
            wait += 1
            print(f"No improvement ({wait}/{patience})")

            if wait >= patience:
                print("Early stopping.")
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model

In [124]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionNet(vector_dim=train_vectors_df.shape[1] ,feature_dim=train_features_df.shape[1]-1, num_classes=2, dropout=0.3).to(device)
model = train_model(model, train_loader, val_loader, device, epochs=50, patience=20, model_path="best_fusion_model.pth")

Epoch   1 | Train Loss 0.6465 | Val Loss 0.6729 | Val F1 0.4163 | LR 0.000100
Validation F1 improved -> model saved.
Epoch   2 | Train Loss 0.5901 | Val Loss 0.5999 | Val F1 0.4163 | LR 0.000100
No improvement (1/20)
Epoch   3 | Train Loss 0.5333 | Val Loss 0.5368 | Val F1 0.4163 | LR 0.000100
No improvement (2/20)
Epoch   4 | Train Loss 0.4819 | Val Loss 0.4987 | Val F1 0.4163 | LR 0.000050
No improvement (3/20)
Epoch   5 | Train Loss 0.4600 | Val Loss 0.4869 | Val F1 0.4398 | LR 0.000050
Validation F1 improved -> model saved.
Epoch   6 | Train Loss 0.4396 | Val Loss 0.4783 | Val F1 0.4518 | LR 0.000050
Validation F1 improved -> model saved.
Epoch   7 | Train Loss 0.4308 | Val Loss 0.4722 | Val F1 0.5038 | LR 0.000050
Validation F1 improved -> model saved.
Epoch   8 | Train Loss 0.4144 | Val Loss 0.4683 | Val F1 0.6018 | LR 0.000050
Validation F1 improved -> model saved.
Epoch   9 | Train Loss 0.3992 | Val Loss 0.4664 | Val F1 0.6332 | LR 0.000050
Validation F1 improved -> model saved